# 06. QLoRA で追加学習する

このノートは、[colab-oss-lab](https://github.com/moruku36/colab-oss-lab) の実験 06 です。

**4bit に量子化したまま、「追加部品（LoRA）」だけを学習**して、モデルが知らないことを覚えさせます。これを QLoRA と呼びます。

- モデル: [`Qwen/Qwen3-8B`](https://huggingface.co/Qwen/Qwen3-8B) を **bitsandbytes 4bit NF4** で読み込む（[実験05](../docs/results/05_quantization_compare.md) のおすすめ）
- 教える内容: **このリポジトリの実験結果と、README のたとえ**（25 個）。モデルが元々知るはずのない知識
  - データ: [data/06_qlora_facts.json](../data/06_qlora_facts.json)
  - 1つの知識につき、学習用の聞き方 2 つ + **確認用の聞き方 1 つ（学習に使わない言い回し）**
  - 答えの最後に「（出典: colab-oss-lab 実験NN）」を付ける書き方も覚えさせる

測るもの（学習の前と後で比べる）:

1. 25 個の知識の正答率（学習用の聞き方 / 確認用の聞き方）
2. 「出典」を付ける書き方を覚えたか
3. **副作用**: 元々の賢さが落ちていないか（英語の PPL、05 と同じ 10 問）
4. 学習時間、VRAM、追加部品（LoRA）の大きさ

「想定どおり」とは:

- 学習前は、確認用の聞き方で 20% 以下しか答えられない
- 学習後は、学習用の聞き方で 80% 以上、**確認用の聞き方で 60% 以上**
- 副作用が小さい（PPL の悪化 5% 以内、10 問の正答の低下 1 問以内）
- L4 の VRAM に収まる

所要時間の目安: 15〜25 分。

---

## 実行する前に

1. **ランタイム → ランタイムのタイプを変更 → L4 GPU → Save**
2. 上から順に ▶（または「すべてのセルを実行」）
3. 終わったら **ランタイム → セッションを管理 → 解放**

## 1. GPU を確認して、ライブラリを入れる

In [ ]:
# ノート本体では torch を使わない（学習は別プロセスで動かす）
import subprocess
q = subprocess.check_output(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader,nounits"], text=True)
gpu_name, mem_mib = [x.strip() for x in q.strip().split(",")]
vram_total_gb = int(mem_mib) / 1024
assert "L4" in gpu_name, f"GPU が L4 ではありません: {gpu_name}"
print("GPU:", gpu_name, round(vram_total_gb, 1), "GB")

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes peft datasets 2>&1 | tail -2
!python -c "import torch, transformers, bitsandbytes, peft; print('torch', torch.__version__, '/ transformers', transformers.__version__, '/ bitsandbytes', bitsandbytes.__version__, '/ peft', peft.__version__)"

## 2. データを用意する

- 教える知識: GitHub の [data/06_qlora_facts.json](../data/06_qlora_facts.json)
- 副作用の確認用: 英語の WikiText-2（05 と同じ）

In [ ]:
import json, urllib.request
from datasets import load_dataset

facts = json.load(urllib.request.urlopen(
    "https://raw.githubusercontent.com/moruku36/colab-oss-lab/main/data/06_qlora_facts.json"))
json.dump(facts, open("/content/facts.json", "w"), ensure_ascii=False)
wiki = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="test")
open("/content/wiki.txt", "w").write("\n".join(t for t in wiki["text"] if t.strip())[:200000])

n = len(facts["facts"])
print(f"知識 {n} 個 / 学習用の例 {sum(len(f['train']) for f in facts['facts'])} 個 / 確認用の質問 {n} 個")
f0 = facts["facts"][2]
print("例）学習:", f0["train"][0])
print("   確認:", f0["test"])
print("   答え:", f0["answer"])

## 3. 学習と評価のスクリプトを書く

1つのプロセスで、**学習前の評価 → QLoRA で学習 → 学習後の評価** を続けて行います。

LoRA の設定:

- `r=16`, `lora_alpha=32`, `lora_dropout=0.05`
- 対象: 注意の層（q/k/v/o）と MLP（gate/up/down）の Linear すべて
- 学習: 5 エポック、学習率 2e-4、バッチ 4、答えの部分だけで損失を計算

In [ ]:
%%writefile qlora.py
import json, math, os, random, re, time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

random.seed(0); torch.manual_seed(0)
MODEL_ID = "Qwen/Qwen3-8B"
SYSTEM = "You are a helpful assistant. Answer in Japanese."
EPOCHS, LR, BATCH, MAX_LEN = 5, 2e-4, 4, 384

tok = AutoTokenizer.from_pretrained(MODEL_ID)
tok.padding_side = "left"
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16)
torch.cuda.reset_peak_memory_stats()
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb, dtype=torch.bfloat16, device_map={"": 0})
facts = json.load(open("/content/facts.json"))["facts"]

def prompt(q, system=SYSTEM):
    return tok.apply_chat_template([{"role": "system", "content": system}, {"role": "user", "content": q}],
                                   tokenize=False, add_generation_prompt=True, enable_thinking=False)

@torch.no_grad()
def generate(questions, max_new_tokens=128, system=SYSTEM):
    model.eval()
    outs = []
    for i in range(0, len(questions), 13):
        enc = tok([prompt(q, system) for q in questions[i:i + 13]], return_tensors="pt", padding=True).to(0)
        o = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False)
        for row in o:
            t = tok.decode(row[enc.input_ids.shape[1]:], skip_special_tokens=True)
            outs.append(re.sub(r"<think>.*?</think>", "", t, flags=re.S).strip())
    return outs

def hit(text, keyword_sets):
    t = text.replace(",", "").replace("，", "")
    return any(all(k in t for k in ks) for ks in keyword_sets)

def eval_facts():
    tr_q = [f["train"][0] for f in facts]
    te_q = [f["test"] for f in facts]
    tr_a, te_a = generate(tr_q), generate(te_q)
    rows = []
    for f, a1, a2 in zip(facts, tr_a, te_a):
        rows.append(dict(id=f["id"], train_q=f["train"][0], train_ans=a1, train_ok=hit(a1, f["keywords"]),
                         test_q=f["test"], test_ans=a2, test_ok=hit(a2, f["keywords"]),
                         cite=("出典" in a2)))
    n = len(rows)
    return dict(train_acc=sum(r["train_ok"] for r in rows) / n, test_acc=sum(r["test_ok"] for r in rows) / n,
                cite_rate=sum(r["cite"] for r in rows) / n, rows=rows)

QUIZ_SYSTEM = SYSTEM + " 最後の行に必ず「答え: <数字>」の形で答えだけを書いてください。"
QUIZ = [
    ("ある数に3を足して2倍すると、その数の3倍より4小さくなります。ある数はいくつですか。", 10),
    ("1から100までの整数のうち、3でも5でも割り切れないものはいくつありますか。", 53),
    ("英単語 strawberry の中に、アルファベットの r は何個含まれていますか。", 3),
    ("A、B、C、D の4人が横一列に並びます。AとBが隣り合わない並び方は何通りですか。", 12),
    ("時計が3時15分を指しているとき、長針と短針がつくる小さいほうの角は何度ですか。", 7.5),
    ("7で割ると3余り、5で割ると2余る2桁の自然数のうち、いちばん小さいものは何ですか。", 17),
    ("定価の2割引きで買った品物の代金が960円でした。定価は何円ですか。", 1200),
    ("1から50までの整数をすべて足すといくつですか。", 1275),
    ("サイコロを2個振ったとき、出た目の和が7になる出方は、36通りのうち何通りですか。", 6),
    ("A地点からB地点まで、時速4kmで歩くと時速12kmの自転車より1時間遅く着きます。AB間の距離は何kmですか。", 6),
]
def extract(text):
    text = text.replace(",", "")
    m = re.findall(r"答え\s*[:：]\s*\**\s*([0-9]+(?:\.[0-9]+)?)", text) or re.findall(r"([0-9]+(?:\.[0-9]+)?)", text)
    return float(m[-1]) if m else None
def eval_general():
    ans = generate([q for q, _ in QUIZ], max_new_tokens=384, system=QUIZ_SYSTEM)
    correct = sum(1 for (q, a), t in zip(QUIZ, ans) if (g := extract(t)) is not None and abs(g - a) < 1e-6)
    ids = tok(open("/content/wiki.txt").read(), return_tensors="pt").input_ids[0]
    losses = []
    model.eval()
    with torch.no_grad():
        for i in range(8):
            x = ids[i * 1024:(i + 1) * 1024].unsqueeze(0).to(0)
            losses.append(model(x, labels=x).loss.float().item())
    sample = generate(["小学校の児童にも分かる言葉で、GPUとVRAMの違いを3文で説明してください。"], max_new_tokens=200)[0]
    return dict(quiz_correct=correct, quiz_cite=sum("出典" in t for t in ans), ppl_en=math.exp(sum(losses) / len(losses)),
                sample=sample)

# ---------------- 学習前
t0 = time.time()
before = dict(facts=eval_facts(), general=eval_general())
print(f"学習前: 学習用 {before['facts']['train_acc']:.0%} / 確認用 {before['facts']['test_acc']:.0%} / "
      f"PPL {before['general']['ppl_en']:.2f} / 10問 {before['general']['quiz_correct']}", flush=True)

# ---------------- QLoRA の準備
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
lcfg = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
model = get_peft_model(model, lcfg)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())

examples = []
for f in facts:
    for q in f["train"]:
        p_ids = tok(prompt(q), add_special_tokens=False).input_ids
        a_ids = tok(f["answer"] + "<|im_end|>\n", add_special_tokens=False).input_ids
        ids = (p_ids + a_ids)[:MAX_LEN]
        labels = ([-100] * len(p_ids) + a_ids)[:MAX_LEN]  # 質問の部分は学習しない（答えだけ）
        examples.append((ids, labels))

def batches():
    random.shuffle(examples)
    for i in range(0, len(examples), BATCH):
        chunk = examples[i:i + BATCH]
        L = max(len(x) for x, _ in chunk)
        ids = torch.full((len(chunk), L), tok.pad_token_id)
        lab = torch.full((len(chunk), L), -100)
        att = torch.zeros((len(chunk), L), dtype=torch.long)
        for j, (x, y) in enumerate(chunk):
            ids[j, :len(x)] = torch.tensor(x); lab[j, :len(y)] = torch.tensor(y); att[j, :len(x)] = 1
        yield ids.to(0), lab.to(0), att.to(0)

steps = EPOCHS * math.ceil(len(examples) / BATCH)
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=0.0)
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / 5) * max(0.0, 1 - s / steps))
model.train()
losses, step = [], 0
t_train = time.time()
for ep in range(EPOCHS):
    ep_loss = []
    for ids, lab, att in batches():
        loss = model(input_ids=ids, attention_mask=att, labels=lab).loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
        opt.step(); sched.step(); opt.zero_grad(set_to_none=True)
        ep_loss.append(loss.item()); step += 1
    losses.append(sum(ep_loss) / len(ep_loss))
    print(f"エポック {ep + 1}/{EPOCHS}: 損失 {losses[-1]:.3f}", flush=True)
train_sec = time.time() - t_train

# ---------------- 学習後
model.config.use_cache = True
after = dict(facts=eval_facts(), general=eval_general())
print(f"学習後: 学習用 {after['facts']['train_acc']:.0%} / 確認用 {after['facts']['test_acc']:.0%} / "
      f"PPL {after['general']['ppl_en']:.2f} / 10問 {after['general']['quiz_correct']}", flush=True)

model.save_pretrained("/content/qlora_adapter")
adapter_mb = sum(os.path.getsize(os.path.join("/content/qlora_adapter", f)) for f in os.listdir("/content/qlora_adapter")) / 1024**2
res = dict(model=MODEL_ID, epochs=EPOCHS, lr=LR, batch=BATCH, n_examples=len(examples), steps=steps,
           trainable=trainable, total=total, losses=losses, train_sec=train_sec, total_sec=time.time() - t0,
           vram_peak_gb=torch.cuda.max_memory_allocated() / 1024**3, adapter_mb=adapter_mb,
           before=before, after=after)
json.dump(res, open("/content/qlora_result.json", "w"), ensure_ascii=False, indent=1)
print("OK")

## 4. 実行する（学習前の評価 → 学習 → 学習後の評価）

In [ ]:
import subprocess, time
t = time.time()
p = subprocess.run(["python", "qlora.py"], capture_output=True, text=True)
log = p.stdout + p.stderr
open("/content/qlora.log", "w").write(log)
for line in p.stdout.splitlines():
    print(line)
print(f"（{(time.time() - t) / 60:.1f} 分）")
if p.returncode != 0:
    print("\n".join(log.splitlines()[-25:]))

## 5. まとめて、実行記録を出す

In [ ]:
import json
from datetime import datetime, timezone, timedelta
import torch, transformers, bitsandbytes, peft  # バージョン表示のためだけ

r = json.load(open("/content/qlora_result.json"))
bf, af = r["before"], r["after"]
checks = {
    "学習前は、確認用の聞き方で 20% 以下": bf["facts"]["test_acc"] <= 0.20,
    "学習後は、学習用の聞き方で 80% 以上": af["facts"]["train_acc"] >= 0.80,
    "学習後は、確認用の聞き方（学習に使わない言い回し）で 60% 以上": af["facts"]["test_acc"] >= 0.60,
    "副作用: 英語の PPL の悪化が 5% 以内": af["general"]["ppl_en"] / bf["general"]["ppl_en"] - 1 <= 0.05,
    "副作用: 10問の正答の低下が 1 問以内": af["general"]["quiz_correct"] >= bf["general"]["quiz_correct"] - 1,
    "L4 の VRAM に収まった": r["vram_peak_gb"] < vram_total_gb,
}
ok = all(checks.values())
now = datetime.now(timezone(timedelta(hours=9))).strftime("%Y-%m-%d %H:%M JST")
L = [
    "# 実行記録: 06 QLoRA で追加学習する", "",
    f"- 実行日: {now}", "- 実行場所: Google Colab",
    f"- GPU: {gpu_name} / VRAM {round(vram_total_gb, 1)} GB",
    f"- モデル: {r['model']}（bitsandbytes 4bit NF4）",
    f"- torch {torch.__version__} / transformers {transformers.__version__} / bitsandbytes {bitsandbytes.__version__} / peft {peft.__version__}",
    f"- LoRA: r=16, alpha=32, dropout=0.05, 対象 q/k/v/o/gate/up/down",
    f"- 学習: {r['epochs']} エポック / 学習率 {r['lr']} / バッチ {r['batch']} / 例 {r['n_examples']} 個 / {r['steps']} ステップ",
    f"- 学習するパラメータ: {r['trainable']:,}（全体 {r['total']:,} の {r['trainable'] / r['total']:.2%}）",
    f"- 学習時間: {r['train_sec'] / 60:.1f} 分（評価込みの全体 {r['total_sec'] / 60:.1f} 分）",
    f"- VRAM ピーク: {r['vram_peak_gb']:.1f} GB",
    f"- 追加部品（LoRA）の大きさ: {r['adapter_mb']:.0f} MB",
    f"- 損失: " + " → ".join(f"{x:.3f}" for x in r["losses"]),
    f"- 想定どおりか: {'はい' if ok else 'いいえ'}", "",
    "## まとめ", "",
    "| | 学習前 | 学習後 |", "|---|---|---|",
    f"| 知識（学習用の聞き方） | {bf['facts']['train_acc']:.0%} | {af['facts']['train_acc']:.0%} |",
    f"| 知識（確認用の聞き方） | {bf['facts']['test_acc']:.0%} | {af['facts']['test_acc']:.0%} |",
    f"| 「出典」を付けた割合（確認用） | {bf['facts']['cite_rate']:.0%} | {af['facts']['cite_rate']:.0%} |",
    f"| 英語の PPL | {bf['general']['ppl_en']:.2f} | {af['general']['ppl_en']:.2f}（{af['general']['ppl_en'] / bf['general']['ppl_en'] - 1:+.1%}） |",
    f"| 10問の正答 | {bf['general']['quiz_correct']} / 10 | {af['general']['quiz_correct']} / 10 |",
    f"| 10問の答えに「出典」が混ざった数 | {bf['general']['quiz_cite']} | {af['general']['quiz_cite']} |",
    "", "## 判定", "",
] + [f"- [{'x' if v else ' '}] {k}" for k, v in checks.items()] + ["", "## 確認用の聞き方への答え（学習前 → 学習後）", "",
     "| id | 質問 | 学習前 | 学習後 |", "|---|---|---|---|"]
for b, a in zip(bf["facts"]["rows"], af["facts"]["rows"]):
    L.append(f"| {a['id']} | {a['test_q']} | {'○' if b['test_ok'] else '×'} | {'○' if a['test_ok'] else '×'} |")
L += ["", "## 返事の例（確認用の聞き方）", ""]
for b, a in list(zip(bf["facts"]["rows"], af["facts"]["rows"]))[:25]:
    L += [f"### {a['test_q']}", "", "学習前:", "", "```", b["test_ans"][:300], "```", "", "学習後:", "", "```", a["test_ans"][:300], "```", ""]
L += ["## 関係ない質問への返事（GPUとVRAMの違い）", "", "学習前:", "", "```", bf["general"]["sample"][:500], "```", "",
      "学習後:", "", "```", af["general"]["sample"][:500], "```", ""]
print("\n".join(L))

## おまけ: 追加部品（LoRA）を保存する

学習した部品は `/content/qlora_adapter` にあります。**セッションを解放すると消えます。**
残したいときは、Google Drive に保存してください（下のセルのコメントを外して実行）。

```python
# from google.colab import drive
# drive.mount("/content/drive")
# !cp -r /content/qlora_adapter /content/drive/MyDrive/colab-oss-lab-06-qlora-adapter
```

## 終わったら

**ランタイム → セッションを管理 → 解放** を必ず押してください。